# VisionFlowAI Part 3

Standalone notebook implementation of the multimodal multi-agent image workflow. It follows the SDD and rubric: Vision Understanding, Prompt Engineering, Image Generation, Critique/Evaluation, integration logs, three test cases, human-evaluation reports, and graceful fallbacks.

Outputs are written to `input_images/`, `generated_outputs/`, and `agent_logs/`.

## Google Colab Demo Setup

Run the next setup cell first in Colab. For a no-key demo, keep `MOCK_MODE=true`; the workflow still executes all agents and creates local transformed output images. For a real provider demo, add `OPENAI_API_KEY` and `REPLICATE_API_TOKEN` in Colab secrets or set them as environment variables, then set `MOCK_MODE=false`.

Recommended demo flow:
1. Runtime -> Run all for the built-in three rubric cases.
2. Confirm outputs in `input_images/`, `generated_outputs/`, and `agent_logs/`.
3. Use the final optional custom-image cell for a real image upload demo.

In [ ]:
import os, subprocess, sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        'pydantic>=2.7',
        'Pillow>=10.0',
        'openai>=1.30',
        'replicate>=0.25',
        'requests>=2.32',
        'python-dotenv>=1.0',
    ])

# Real API demo defaults for Colab. Set USE_REAL_APIS=False only for offline/mock testing.
USE_REAL_APIS = True

if USE_REAL_APIS and IN_COLAB:
    from google.colab import userdata
    for key in ('OPENAI_API_KEY', 'REPLICATE_API_TOKEN'):
        secret = userdata.get(key)
        if secret:
            os.environ[key] = secret

if USE_REAL_APIS:
    os.environ['MOCK_MODE'] = 'false'
    os.environ.setdefault('OPENAI_VISION_MODEL', 'gpt-4o')
    os.environ.setdefault('OPENAI_TEXT_MODEL', 'gpt-4o')
    # Use the public model slug by default. Pin a version only if your Replicate account has access to it.
    os.environ.setdefault('REPLICATE_SDXL_IMG2IMG_MODEL', 'stability-ai/sdxl')
else:
    os.environ.setdefault('MOCK_MODE', 'true')

os.environ.setdefault('CLIP_ALLOW_DOWNLOAD', 'false')
print('Colab:', IN_COLAB)
print('USE_REAL_APIS:', USE_REAL_APIS)
print('MOCK_MODE:', os.environ.get('MOCK_MODE'))


In [ ]:
from __future__ import annotations

import base64, csv, json, math, os, shutil, time
from pathlib import Path
from typing import Any, Literal

from PIL import Image, ImageChops, ImageDraw, ImageEnhance, ImageOps, ImageStat
from pydantic import BaseModel, Field, field_validator

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

INPUT_DIR = Path('input_images')
OUTPUT_DIR = Path('generated_outputs')
LOG_DIR = Path('agent_logs')
for directory in (INPUT_DIR, OUTPUT_DIR, LOG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

class VisionContext(BaseModel):
    '''Requirement 3.1: caption, objects, scene, and visual QA answers.'''
    caption: str
    objects: list[str] = Field(default_factory=list)
    scene_description: str
    qa_answers: dict[str, str] = Field(default_factory=dict)

class PromptData(BaseModel):
    '''Requirement 3.2: refined prompt and transformation mode.'''
    refined_prompt: str
    transformation_mode: Literal['enhancement', 'stylization', 'variation', 'scene_edit']
    added_visual_details: list[str] = Field(default_factory=list)

class GenerationData(BaseModel):
    '''Requirement 3.3: generated image and generation config.'''
    generated_image_path: str
    generation_config: dict[str, Any] = Field(default_factory=dict)

class EvaluationMetrics(BaseModel):
    '''Requirement 3.4: critique scores and automatic signal.'''
    visual_relevance_score: int = Field(ge=1, le=10)
    prompt_faithfulness_score: int = Field(ge=1, le=10)
    transformation_quality: int = Field(ge=1, le=10)
    clip_similarity_score: float = Field(ge=-1.0, le=1.0)
    critique_summary: str
    decision: Literal['ACCEPT', 'REVISE']

class HumanEvaluationTemplate(BaseModel):
    visual_relevance_1_to_5: int | None = None
    prompt_faithfulness_1_to_5: int | None = None
    transformation_quality_1_to_5: int | None = None
    human_notes: str = ''

class WorkflowState(BaseModel):
    '''Requirement 3.5: shared state for explicit agent communication.'''
    original_image_path: str
    user_instruction: str
    visual_questions: list[str]
    vision_context: VisionContext | None = None
    prompt_data: PromptData | None = None
    generation_data: GenerationData | None = None
    evaluation_metrics: EvaluationMetrics | None = None
    human_evaluation_template: HumanEvaluationTemplate | None = None
    error_log: list[str] = Field(default_factory=list)

    @field_validator('original_image_path')
    @classmethod
    def image_path_must_exist(cls, value: str) -> str:
        if not Path(value).exists():
            raise ValueError(f'Input image does not exist: {value}')
        return value

    @field_validator('visual_questions')
    @classmethod
    def require_two_questions(cls, value: list[str]) -> list[str]:
        cleaned = [q.strip() for q in value if q.strip()]
        if len(cleaned) < 2:
            raise ValueError('At least two visual questions are required.')
        return cleaned

    @property
    def generated_image_path(self) -> str | None:
        return self.generation_data.generated_image_path if self.generation_data else None

    @property
    def critique_decision(self) -> str | None:
        return self.evaluation_metrics.decision if self.evaluation_metrics else None

for model in (VisionContext, PromptData, GenerationData, EvaluationMetrics, HumanEvaluationTemplate, WorkflowState):
    model.model_rebuild(_types_namespace=globals())

class VisionAgent:
    '''Requirement 3.1: VLM-style image understanding with API and local fallback.'''
    def __init__(self, model: str | None = None) -> None:
        self.model = model or os.getenv('OPENAI_VISION_MODEL', 'gpt-4o')

    def execute(self, image_path: str, visual_questions: list[str]) -> VisionContext:
        if not self._is_readable_image(image_path):
            raise ValueError('[Error] Input image quality too low for feature extraction')
        if os.getenv('OPENAI_API_KEY'):
            return self._execute_with_openai(image_path, visual_questions)
        return self._execute_local(image_path, visual_questions)

    def _execute_with_openai(self, image_path: str, visual_questions: list[str]) -> VisionContext:
        from openai import OpenAI
        encoded = base64.b64encode(Path(image_path).read_bytes()).decode('utf-8')
        client = OpenAI()
        prompt = 'Analyze the image. Return only valid JSON with required keys caption, objects, scene_description, qa_answers. qa_answers must map each exact question to an answer. Questions: ' + json.dumps(visual_questions)
        try:
            response = client.chat.completions.create(model=self.model, response_format={'type': 'json_object'}, messages=[{'role': 'user', 'content': [{'type': 'text', 'text': prompt}, {'type': 'image_url', 'image_url': {'url': f'data:image/png;base64,{encoded}'}}]}])
            data = json.loads(response.choices[0].message.content or '{}')
            return VisionContext.model_validate(data)
        except Exception as exc:
            print('[Vision Agent Warning] OpenAI returned invalid vision JSON or failed; using local fallback:', exc)
            return self._execute_local(image_path, visual_questions)

    def _execute_local(self, image_path: str, visual_questions: list[str]) -> VisionContext:
        with Image.open(image_path) as image:
            rgb = image.convert('RGB')
            width, height = rgb.size
            stat = ImageStat.Stat(rgb)
            brightness = sum(stat.mean) / 3
        orientation = 'landscape' if width >= height else 'portrait'
        light = 'dark' if brightness < 85 else 'bright' if brightness > 180 else 'balanced'
        tokens = [t.lower() for t in Path(image_path).stem.replace('-', '_').split('_')]
        objects = self._infer_objects(tokens, orientation, light)
        caption = f'A {light} {orientation} image with visible subject detail.'
        object_text = ', '.join(objects[:3])
        scene = f'The image appears to be a {orientation} composition with {light} exposure and visible {object_text}.'
        qa = {question: self._answer(question, objects, light, tokens) for question in visual_questions}
        return VisionContext(caption=caption, objects=objects, scene_description=scene, qa_answers=qa)

    @staticmethod
    def _is_readable_image(image_path: str) -> bool:
        try:
            with Image.open(image_path) as image:
                image.verify()
            with Image.open(image_path) as image:
                w, h = image.size
            return w >= 16 and h >= 16
        except Exception:
            return False

    @staticmethod
    def _infer_objects(tokens: list[str], orientation: str, light: str) -> list[str]:
        mapping = {'street': ['street', 'cars', 'buildings', 'road'], 'cat': ['cat', 'fur', 'eyes', 'background'], 'landscape': ['landscape', 'sky', 'terrain', 'horizon'], 'dark': ['shadows', 'low light']}
        objects = []
        for token in tokens:
            objects.extend(mapping.get(token, []))
        return list(dict.fromkeys(objects or ['main subject', 'background', f'{orientation} frame', f'{light} lighting']))

    @staticmethod
    def _answer(question: str, objects: list[str], light: str, tokens: list[str]) -> str:
        q = question.lower()
        if 'how many cars' in q: return 'Several cars are visible or suggested by the scene.' if 'cars' in objects else 'No cars are apparent.'
        if 'weather' in q: return 'The weather is not directly visible; lighting appears balanced.'
        if 'animal' in q: return 'A cat.' if 'cat' in objects else 'No specific animal is identifiable.'
        if 'color' in q and ('fur' in q or 'cat' in q): return 'The fur appears warm and neutral in tone.'
        if 'indoors' in q or 'outdoors' in q: return 'Outdoors.' if {'street', 'landscape', 'sky'} & set(objects + tokens) else 'Unclear.'
        if 'people' in q: return 'No people are clearly visible.'
        object_text = ', '.join(objects[:3])
        return f'The image suggests {object_text} with {light} lighting.'

class PromptAgent:
    '''Requirement 3.2: merges user instruction and visual context into an SDXL-ready prompt.'''
    def __init__(self, model: str | None = None) -> None:
        self.model = model or os.getenv('OPENAI_TEXT_MODEL', 'gpt-4o')

    def execute(self, user_instruction: str, vision_context: VisionContext) -> PromptData:
        if os.getenv('OPENAI_API_KEY'):
            return self._execute_with_openai(user_instruction, vision_context)
        return self._execute_local(user_instruction, vision_context)

    def _execute_with_openai(self, user_instruction: str, vision_context: VisionContext) -> PromptData:
        from openai import OpenAI
        client = OpenAI()
        system = 'Return only valid JSON with refined_prompt as a string, transformation_mode as one of enhancement/stylization/variation/scene_edit, and added_visual_details as an array of strings. Never return added_visual_details as a plain string.'
        payload = {'user_instruction': user_instruction, 'vision_context': vision_context.model_dump()}
        try:
            response = client.chat.completions.create(model=self.model, response_format={'type': 'json_object'}, messages=[{'role': 'system', 'content': system}, {'role': 'user', 'content': json.dumps(payload)}])
            data = json.loads(response.choices[0].message.content or '{}')
            if isinstance(data.get('added_visual_details'), str):
                data['added_visual_details'] = [data['added_visual_details']]
            if data.get('transformation_mode') not in {'enhancement', 'stylization', 'variation', 'scene_edit'}:
                data['transformation_mode'] = self._classify_mode(user_instruction)
            return PromptData.model_validate(data)
        except Exception as exc:
            print('[Prompt Agent Warning] OpenAI returned invalid prompt JSON or failed; using local fallback:', exc)
            return self._execute_local(user_instruction, vision_context)

    def _execute_local(self, user_instruction: str, vision_context: VisionContext) -> PromptData:
        instruction = user_instruction.strip().rstrip('.') or 'Enhance the image while preserving the original composition'
        mode = self._classify_mode(instruction)
        visible = ', '.join(vision_context.objects[:6])
        details = [vision_context.caption, vision_context.scene_description, f'Key visible elements: {visible}.']
        detail_text = ' '.join(details)
        prompt = f'{instruction}. Preserve the original composition, subject identity, spatial layout, and important objects. Use these visual anchors: {detail_text} Create a coherent, high-quality image with natural lighting, clean detail, and no unwanted artifacts.'
        return PromptData(refined_prompt=prompt, transformation_mode=mode, added_visual_details=details)

    @staticmethod
    def _classify_mode(instruction: str) -> str:
        text = instruction.lower()
        if any(w in text for w in ['cyberpunk', 'painting', 'style', 'watercolor', 'anime']): return 'stylization'
        if any(w in text for w in ['enhance', 'lighting', 'sharpen', 'golden hour', 'pop']): return 'enhancement'
        if any(w in text for w in ['change', 'replace', 'add', 'remove', 'edit']): return 'scene_edit'
        if any(w in text for w in ['variation', 'different', 'recreate']): return 'variation'
        return 'enhancement'

class GenerationAgent:
    '''Requirement 3.3: image-to-image generation with Replicate path and mock fallback.'''
    MODE_DENOISING = {'enhancement': 0.3, 'stylization': 0.6, 'scene_edit': 0.7, 'variation': 0.8}
    def __init__(self, output_dir: Path = OUTPUT_DIR) -> None:
        self.output_dir = Path(output_dir); self.output_dir.mkdir(parents=True, exist_ok=True)
        self.mock_mode = os.getenv('MOCK_MODE', 'true').lower() in {'1', 'true', 'yes'}
        self.model = os.getenv('REPLICATE_SDXL_IMG2IMG_MODEL', 'stability-ai/sdxl')

    def execute(self, original_image_path: str, refined_prompt: str, transformation_mode: str, simplified_retry: bool = False) -> GenerationData:
        denoise = self.MODE_DENOISING.get(transformation_mode, 0.3)
        if simplified_retry: denoise = min(denoise, 0.35)
        if self.mock_mode or not os.getenv('REPLICATE_API_TOKEN'):
            path = self._mock_image(original_image_path, transformation_mode)
        else:
            path = self._replicate_image(original_image_path, refined_prompt, denoise)
        config = {'prompt_used': refined_prompt, 'model': 'mock-sdxl-img2img' if self.mock_mode else self.model, 'inference_steps': 50, 'denoising_strength': denoise, 'mode': transformation_mode}
        return GenerationData(generated_image_path=str(path), generation_config=config)

    def _mock_image(self, original_image_path: str, mode: str) -> Path:
        path = self.output_dir / f'gen_image_{int(time.time() * 1000)}.png'
        with Image.open(original_image_path) as image:
            rgb = image.convert('RGB')
            if mode == 'stylization': out = ImageOps.posterize(ImageEnhance.Color(rgb).enhance(1.9), bits=4)
            elif mode == 'variation': out = ImageOps.mirror(ImageEnhance.Contrast(rgb).enhance(1.15))
            else: out = ImageEnhance.Sharpness(ImageEnhance.Brightness(rgb).enhance(1.18)).enhance(1.4)
            out.save(path)
        return path

    def _replicate_image(self, original_image_path: str, prompt: str, denoise: float) -> Path:
        import replicate, requests
        path = self.output_dir / f'gen_image_{int(time.time() * 1000)}.png'
        last_error = None
        for attempt in range(3):
            try:
                with open(original_image_path, 'rb') as init_image:
                    result = replicate.run(self.model, input={'image': init_image, 'prompt': prompt, 'num_inference_steps': 50, 'prompt_strength': denoise, 'num_outputs': 1})
                url = result[0] if isinstance(result, list) else result
                response = requests.get(str(url), timeout=60); response.raise_for_status(); path.write_bytes(response.content)
                return path
            except Exception as exc:
                last_error = exc
                message = str(exc)
                if '429' in message or 'throttled' in message.lower() or 'rate limit' in message.lower():
                    wait_seconds = 12 * (attempt + 1)
                    print(f'[Generation Agent Warning] Replicate rate limited; waiting {wait_seconds}s before retry {attempt + 2}/3')
                    time.sleep(wait_seconds)
                    continue
                raise
        raise last_error

class CritiqueAgent:
    '''Requirement 3.4: evaluates image relevance, faithfulness, quality, CLIP signal, and human rubric.'''
    def __init__(self, clip_model_name: str = 'openai/clip-vit-base-patch32') -> None:
        self.clip_model_name = clip_model_name
        self._clip_model = None
        self._clip_processor = None

    def execute(self, original_image_path: str, generated_image_path: str, refined_prompt: str, transformation_mode: str):
        clip_score = self._clip_similarity(generated_image_path, refined_prompt)
        if os.getenv('OPENAI_API_KEY'):
            metrics = self._execute_with_openai(original_image_path, generated_image_path, refined_prompt, transformation_mode, clip_score)
        else:
            metrics = self._execute_local(original_image_path, generated_image_path, refined_prompt, transformation_mode, clip_score)
        return metrics, HumanEvaluationTemplate()

    def _execute_with_openai(self, original_image_path: str, generated_image_path: str, refined_prompt: str, transformation_mode: str, clip_score: float) -> EvaluationMetrics:
        try:
            from openai import OpenAI
            client = OpenAI()
            original_b64 = base64.b64encode(Path(original_image_path).read_bytes()).decode('utf-8')
            generated_b64 = base64.b64encode(Path(generated_image_path).read_bytes()).decode('utf-8')
            prompt = 'Compare the original and generated images against the generation prompt. Return only JSON with visual_relevance_score, prompt_faithfulness_score, transformation_quality, critique_summary, and decision. Scores must be integers 1-10 and decision must be ACCEPT or REVISE. Prompt: ' + refined_prompt
            response = client.chat.completions.create(model=os.getenv('OPENAI_VISION_MODEL', 'gpt-4o'), response_format={'type': 'json_object'}, messages=[{'role': 'user', 'content': [{'type': 'text', 'text': prompt}, {'type': 'image_url', 'image_url': {'url': f'data:image/png;base64,{original_b64}'}}, {'type': 'image_url', 'image_url': {'url': f'data:image/png;base64,{generated_b64}'}}]}])
            data = json.loads(response.choices[0].message.content or '{}')
            data['clip_similarity_score'] = clip_score
            data['decision'] = data.get('decision', 'ACCEPT' if clip_score >= 0.15 else 'REVISE')
            return EvaluationMetrics.model_validate(data)
        except Exception as exc:
            return self._execute_local(original_image_path, generated_image_path, refined_prompt, transformation_mode, clip_score, note=f'OpenAI critique fallback: {exc}')

    def _execute_local(self, original_image_path: str, generated_image_path: str, refined_prompt: str, transformation_mode: str, clip_score: float, note: str = '') -> EvaluationMetrics:
        similarity = self._pixel_similarity(original_image_path, generated_image_path)
        visual = max(1, min(10, round(10 - abs(similarity - 0.75) * 12)))
        faithful = 9 if 'preserve' in refined_prompt.lower() else 6
        quality = max(6, min(10, round(5 + (1 - similarity) * 4)))
        decision = 'ACCEPT' if visual >= 6 and faithful >= 6 and quality >= 6 else 'REVISE'
        summary = f'The image preserves source structure while applying {transformation_mode}. Automatic signal: {clip_score:.3f}.'
        if note:
            summary += ' ' + note
        return EvaluationMetrics(visual_relevance_score=visual, prompt_faithfulness_score=faithful, transformation_quality=quality, clip_similarity_score=clip_score, critique_summary=summary, decision=decision)

    @staticmethod
    def _pixel_similarity(a: str, b: str) -> float:
        with Image.open(a) as original, Image.open(b) as generated:
            diff = ImageChops.difference(original.convert('RGB').resize((128, 128)), generated.convert('RGB').resize((128, 128)))
            rms = math.sqrt(sum(v ** 2 for v in ImageStat.Stat(diff).rms) / 3)
        return max(0.0, min(1.0, 1 - rms / 255))

    def _clip_similarity(self, image_path: str, prompt: str) -> float:
        try:
            import torch
            from transformers import CLIPModel, CLIPProcessor
            allow_download = os.getenv('CLIP_ALLOW_DOWNLOAD', 'false').lower() in {'1', 'true', 'yes'}
            if self._clip_model is None or self._clip_processor is None:
                self._clip_model = CLIPModel.from_pretrained(self.clip_model_name, local_files_only=not allow_download)
                self._clip_processor = CLIPProcessor.from_pretrained(self.clip_model_name, local_files_only=not allow_download)
            image = Image.open(image_path).convert('RGB')
            inputs = self._clip_processor(text=[prompt], images=image, return_tensors='pt', padding=True, truncation=True, max_length=77)
            with torch.no_grad():
                outputs = self._clip_model(**inputs)
            return float(torch.nn.functional.cosine_similarity(outputs.image_embeds, outputs.text_embeds).item())
        except Exception:
            return self._fallback_signal(image_path, prompt)

    @staticmethod
    def _fallback_signal(image_path: str, prompt: str) -> float:
        with Image.open(image_path) as image:
            brightness = sum(ImageStat.Stat(image.convert('RGB')).mean) / (3 * 255)
        return round(0.15 + 0.25 * brightness + 0.2 * min(len(prompt.split()) / 80, 1.0), 3)

class Orchestrator:
    '''Requirement 3.5: sequential multi-agent collaboration with verbose logs and fallbacks.'''
    def __init__(self) -> None:
        self.vision = VisionAgent(); self.prompt = PromptAgent(); self.generation = GenerationAgent(); self.critique = CritiqueAgent()

    def run(self, image_path: str, instruction: str, questions: list[str], verbose: bool = True, run_id: str = 'workflow') -> WorkflowState:
        state = WorkflowState(original_image_path=image_path, user_instruction=instruction, visual_questions=questions)
        try:
            state.vision_context = self.vision.execute(image_path, questions); self._log(run_id, 'Vision Agent Output', state.vision_context.model_dump(), verbose)
            state.prompt_data = self.prompt.execute(instruction, state.vision_context); self._log(run_id, 'Prompt Agent Output', state.prompt_data.model_dump(), verbose)
            try:
                state.generation_data = self.generation.execute(image_path, state.prompt_data.refined_prompt, state.prompt_data.transformation_mode)
            except Exception as exc:
                state.error_log.append(f'Generation failed; retrying with sanitized prompt: {exc}')
                state.generation_data = self.generation.execute(image_path, state.prompt_data.refined_prompt, state.prompt_data.transformation_mode, simplified_retry=True)
            self._log(run_id, 'Generation Agent Output', state.generation_data.model_dump(), verbose)
            metrics, template = self.critique.execute(image_path, state.generation_data.generated_image_path, state.prompt_data.refined_prompt, state.prompt_data.transformation_mode)
            state.evaluation_metrics = metrics; state.human_evaluation_template = template
            self._log(run_id, 'Critique Agent Output', {**metrics.model_dump(), 'human_evaluation_template': template.model_dump()}, verbose)
        except Exception as exc:
            state.error_log.append(str(exc)); self._log(run_id, 'Error', {'message': str(exc)}, verbose)
        return state

    @staticmethod
    def _log(run_id: str, label: str, payload: dict[str, Any], verbose: bool) -> None:
        if verbose:
            print(f'[{label}]'); print(json.dumps(payload, indent=2))
        safe = label.lower().replace(' ', '_')
        (LOG_DIR / f'{run_id}_{safe}.json').write_text(json.dumps(payload, indent=2), encoding='utf-8')


In [ ]:
def draw_street(path: Path) -> None:
    image = Image.new('RGB', (640, 360), '#9ec8e8'); draw = ImageDraw.Draw(image)
    draw.rectangle((0, 220, 640, 360), fill='#555555'); draw.rectangle((0, 170, 640, 225), fill='#7fb069')
    for x in range(30, 620, 110):
        draw.rectangle((x, 110, x + 55, 220), fill='#b9bec8'); draw.rectangle((x + 10, 125, x + 25, 145), fill='#eaf6ff')
    for x, color in [(70, '#d7263d'), (210, '#f4d35e'), (360, '#1b998b'), (500, '#2e86ab')]:
        draw.rectangle((x, 250, x + 80, 292), fill=color); draw.ellipse((x + 10, 285, x + 28, 303), fill='#111111'); draw.ellipse((x + 55, 285, x + 73, 303), fill='#111111')
    image.save(path)

def draw_cat(path: Path) -> None:
    image = Image.new('RGB', (512, 512), '#6f7d8c'); draw = ImageDraw.Draw(image)
    draw.ellipse((125, 135, 385, 405), fill='#d2a679'); draw.polygon([(155, 170), (195, 70), (235, 175)], fill='#d2a679'); draw.polygon([(275, 175), (320, 70), (360, 170)], fill='#d2a679')
    draw.ellipse((205, 240, 235, 270), fill='#1d3557'); draw.ellipse((285, 240, 315, 270), fill='#1d3557'); draw.polygon([(255, 280), (240, 305), (270, 305)], fill='#ff8fab')
    for y in (285, 305, 325):
        draw.line((125, y, 220, y - 10), fill='#3d2b1f', width=2); draw.line((292, y - 10, 385, y), fill='#3d2b1f', width=2)
    image.save(path)

def draw_landscape(path: Path) -> None:
    image = Image.new('RGB', (640, 360), '#101728'); draw = ImageDraw.Draw(image)
    draw.rectangle((0, 0, 640, 180), fill='#17203a'); draw.polygon([(0, 250), (160, 110), (320, 250)], fill='#1f2f24'); draw.polygon([(230, 250), (445, 95), (640, 250)], fill='#263a2a')
    draw.rectangle((0, 250, 640, 360), fill='#152718'); draw.ellipse((500, 45, 540, 85), fill='#f0c36a'); image.save(path)

images = {'busy_street': INPUT_DIR / 'busy_street.png', 'cat': INPUT_DIR / 'cat.png', 'dark_landscape': INPUT_DIR / 'dark_landscape.png'}
draw_street(images['busy_street']); draw_cat(images['cat']); draw_landscape(images['dark_landscape'])

cases = [
    {'case_id': 'A', 'name': 'Captioning and VQA focus', 'image': images['busy_street'], 'instruction': 'Just recreate this exactly.', 'questions': ['How many cars are visible?', 'What is the weather like?']},
    {'case_id': 'B', 'name': 'Style-guided transformation', 'image': images['cat'], 'instruction': 'Make this look like a cyberpunk digital painting.', 'questions': ['What animal is this?', 'What color is its fur?']},
    {'case_id': 'C', 'name': 'Prompt-based enhancement', 'image': images['dark_landscape'], 'instruction': 'Enhance the lighting, make it golden hour, and make the colors pop.', 'questions': ['Is this indoors or outdoors?', 'Are there any people?']},
]

rows = []
orchestrator = Orchestrator()
for case in cases:
    run_id = 'case_' + case['case_id'].lower()
    state = orchestrator.run(str(case['image']), case['instruction'], case['questions'], verbose=True, run_id=run_id)
    metrics = state.evaluation_metrics; template = state.human_evaluation_template
    rows.append({'case_id': case['case_id'], 'name': case['name'], 'input_image': str(case['image']), 'generated_image': state.generated_image_path or '', 'decision': state.critique_decision or '', 'visual_relevance_score': str(metrics.visual_relevance_score if metrics else ''), 'prompt_faithfulness_score': str(metrics.prompt_faithfulness_score if metrics else ''), 'transformation_quality': str(metrics.transformation_quality if metrics else ''), 'clip_similarity_score': str(metrics.clip_similarity_score if metrics else ''), 'human_visual_relevance_1_to_5': str(template.visual_relevance_1_to_5 if template else ''), 'human_prompt_faithfulness_1_to_5': str(template.prompt_faithfulness_1_to_5 if template else ''), 'human_transformation_quality_1_to_5': str(template.transformation_quality_1_to_5 if template else ''), 'human_notes': template.human_notes if template else '', 'errors': ' | '.join(state.error_log)})

csv_path = LOG_DIR / 'human_evaluation_results.csv'
with csv_path.open('w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=list(rows[0].keys())); writer.writeheader(); writer.writerows(rows)

md_lines = ['# Human Evaluation Results', '', '| Case | Name | Generated Image | Decision | Visual | Prompt | Quality | CLIP | Notes |', '| --- | --- | --- | --- | ---: | ---: | ---: | ---: | --- |']
for row in rows:
    md_lines.append('| {case_id} | {name} | {generated_image} | {decision} | {visual_relevance_score} | {prompt_faithfulness_score} | {transformation_quality} | {clip_similarity_score} | {human_notes} |'.format(**row))
(LOG_DIR / 'human_evaluation_results.md').write_text('\n'.join(md_lines) + '\n', encoding='utf-8')
rows


## Optional Real Image Demo

Use this cell during the demo to upload one real image and run the same workflow on it. Set `RUN_CUSTOM_DEMO = True`, then run the cell. In Colab it opens a file picker; outside Colab it prompts for a local path.

In [ ]:
RUN_CUSTOM_DEMO = False

if RUN_CUSTOM_DEMO:
    if 'google.colab' in sys.modules:
        from google.colab import files
        uploaded = files.upload()
        custom_image_path = next(iter(uploaded.keys()))
    else:
        custom_image_path = input('Enter local image path: ').strip()

    custom_instruction = 'Enhance this image with cinematic golden-hour lighting while preserving the main subject.'
    custom_questions = [
        'What is the main subject in the image?',
        'What is the background or setting?',
    ]

    custom_state = Orchestrator().run(
        image_path=custom_image_path,
        instruction=custom_instruction,
        questions=custom_questions,
        verbose=True,
        run_id='custom_demo',
    )

    custom_result_path = LOG_DIR / 'custom_demo_workflow_result.json'
    custom_result_path.write_text(json.dumps(custom_state.model_dump(), indent=2), encoding='utf-8')
    print('Generated image:', custom_state.generated_image_path)
    print('Decision:', custom_state.critique_decision)
    print('Full result JSON:', custom_result_path)
